In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

# Load raw data
df = pd.read_csv("military_raw_data.csv")

# 1. Clean text columns
def clean_text(val):
    if isinstance(val, str):
        return (
            val.replace(",", "")
               .replace("%", "")
               .replace("+", "")
               .replace("$", "")
               .strip()
        )
    return val

df = df.map(clean_text)

# 2. Convert empty strings to NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 3. Convert metrics to numeric (except Country)
for col in df.columns:
    if col.lower() != "country":
        df[col] = pd.to_numeric(df[col], errors="ignore")

# 4. Handle missing values (numeric → mean)
for col in df.select_dtypes(include=["float64", "int64"]).columns:
    df[col] = df[col].fillna(df[col].mean())

# 5. Label Encoding (except Country)
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == "object" and col.lower() != "country":
        df[col] = le.fit_transform(df[col])

# Save cleaned dataset
df.to_csv("military_cleaned.csv", index=False)

# ============================
# MIN-MAX SCALING (0 to 1)
# ============================
minmax_df = df.copy()
scaler = MinMaxScaler()

numeric_cols = minmax_df.select_dtypes(include=["float64", "int64"]).columns
minmax_df[numeric_cols] = scaler.fit_transform(minmax_df[numeric_cols])

minmax_df.to_csv("military_minmax_scaled.csv", index=False)

# ============================
# STANDARDIZATION (Z-score)
# ============================
std_df = df.copy()
std_scaler = StandardScaler()

std_df[numeric_cols] = std_scaler.fit_transform(std_df[numeric_cols])

std_df.to_csv("military_standardized.csv", index=False)

# Preview
minmax_df.head(), std_df.head()


C:\Users\bhara\AppData\Local\Temp\ipykernel_23972\2465076292.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


(         Country      Rank  total_population_by_country  \
 0  United States  0.000000                     0.241468   
 1         Russia  0.006944                     0.099285   
 2          China  0.013889                     1.000000   
 3          India  0.020833                     0.995819   
 4    South Korea  0.027778                     0.036558   
 
    available_military_manpower  manpower_fit_for_military_service  \
 0                     0.196822                           0.199049   
 1                     0.090203                           0.073609   
 2                     1.000000                           1.000000   
 3                     0.866718                           0.833958   
 4                     0.033974                           0.033987   
 
    manpower_reaching_military_age_annually  active_military_manpower  \
 0                                 0.185515                  0.652580   
 1                                 0.052835                  0.648649 